# CodeGen Capstone — Week 1 baselines + Mentor dataset extension

**Repo:** https://github.com/crav710/codgen  
**Colab branch:** `mentor-extension` (set `CODEGEN_GIT_BRANCH` env var to override)

| Part | What it does |
|------|----------------|
| **1** | Clone/pull from GitHub, install deps, Java, GPU |
| **2** | K=0 baselines (Qwen 1.5B vs 7B) |
| **3** | Mentor task: extend datasets (NL + PL1 + PL2) via `run_extension` |

**Before running:** Runtime → **T4 GPU**

## Part 1 — Setup (run once per Colab session)

In [ ]:
# 1.1 — clone or pull from GitHub (single source of truth)
import os, sys, subprocess

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

GIT_REPO   = os.environ.get('CODEGEN_GIT_REPO',   'https://github.com/crav710/codgen.git')
GIT_BRANCH = os.environ.get('CODEGEN_GIT_BRANCH', 'mentor-extension')
PROJECT_DIR = os.environ.get('CODEGEN_PROJECT_DIR', '/content/drive/MyDrive/codegen_week1')
os.environ['CODEGEN_PROJECT_DIR'] = PROJECT_DIR
os.environ['CODEGEN_DATA_DIR']    = PROJECT_DIR

parent = os.path.dirname(PROJECT_DIR)
os.makedirs(parent, exist_ok=True)

if os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
    print(f'Repo exists — checking out {GIT_BRANCH} and pulling latest...')
    subprocess.run(['git', 'fetch', 'origin'], cwd=PROJECT_DIR, check=True)
    subprocess.run(['git', 'checkout', GIT_BRANCH], cwd=PROJECT_DIR, check=True)
    subprocess.run(['git', 'pull', 'origin', GIT_BRANCH], cwd=PROJECT_DIR, check=True)
else:
    print(f'Cloning {GIT_REPO} (branch={GIT_BRANCH}) ...')
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, GIT_REPO, PROJECT_DIR], check=True)

%cd {PROJECT_DIR}
if PROJECT_DIR in sys.path:
    sys.path.remove(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

# clear stale lib cache when re-pulling
for _k in list(sys.modules):
    if _k == 'lib' or _k.startswith('lib.'):
        del sys.modules[_k]

rev = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=PROJECT_DIR, text=True).strip()
print('Project dir:', os.getcwd())
print('Git branch:', branch, '| commit:', rev)
assert os.path.isfile('lib/ast_compare.py'), 'lib/ast_compare.py missing — wrong branch?'
assert os.path.isfile('lib/extend.py'),     'lib/extend.py missing — wrong branch?'
!ls

/content/drive/MyDrive/codegen_week1


In [ ]:
# 1.2 — install OpenJDK 17 (for HumanEval-X Java scoring)
!apt-get update -qq
!apt-get install -y -q openjdk-17-jdk-headless > /dev/null
!java -version
!javac -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)
javac 17.0.19


In [ ]:
# 1.3 — install Python deps (must run from project dir)
import os
PROJECT_DIR = os.environ.get('CODEGEN_PROJECT_DIR', '/content/drive/MyDrive/codegen_week1')
req = os.path.join(PROJECT_DIR, 'requirements.txt')
assert os.path.isfile(req), f'Missing {req} — run cell 1.1 first'
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print('Installed from:', req)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00


In [ ]:
# 1.4 — mount Drive for persistence between sessions
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.5 — set base data dir, optionally enable smoke mode
import os, sys
PROJECT_DIR = os.environ.get('CODEGEN_PROJECT_DIR', '/content/drive/MyDrive/codegen_week1')
%cd {PROJECT_DIR}
sys.path.insert(0, PROJECT_DIR)
os.environ['CODEGEN_DATA_DIR'] = PROJECT_DIR
# For first dry run
# os.environ['CODEGEN_SMOKE'] = '1'
os.environ['CODEGEN_SEED']   = '42'
os.environ['CODEGEN_SAMPLE'] = '50'

import config
config.ensure_dirs()
print('Data dir:    ', config.DATA_DIR)
print('Smoke mode:  ', config.SMOKE)
print('Problem cap: ', config.BENCHMARK_LIMIT)

Data dir:     /content/drive/MyDrive/codegen_week1
Smoke mode:   False
Problem cap:  50


In [ ]:
# 1.6 — download JUnit5 console-launcher JAR (for HumanEval-X scoring)
from lib.models import setup_java_runtime
setup_java_runtime()

In [ ]:
# 1.7 — smoke-test the Java sandbox
from lib.execution import smoke_test_java
result = smoke_test_java()
assert result['passed'], f'Java sandbox smoke-test FAILED: {result}'
print('Java sandbox OK:', result)

Java sandbox OK: {'passed': True, 'error': None}


In [ ]:
# 1.8 — GPU info
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Runtime -> Change runtime type -> T4 GPU.')
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name}, {p.total_memory/1e9:.1f} GB')

GPU: Tesla T4, 15.6 GB


## K=0 baselines



In [ ]:
# 2.0  clear any leftover GPU memory
import json, time, gc
import pandas as pd
import torch
from tqdm.auto import tqdm

from lib import models, prompts, execution

# if you re-ran cells without restarting the kernel, leftover model
# tensors are still pinned to GPU. Clear them before loading the next model.
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f'GPU free before model load: {free/1e9:.2f} / {total/1e9:.2f} GB')
if free / total < 0.5:
    print('WARNING: less than 50% of VRAM is free. Restart the kernel '
          '(Runtime -> Restart runtime) before loading models, or you will OOM.')

GPU free before model load: 15.53 / 15.64 GB


### 2.1 Benchmark loaders



In [ ]:
from lib.benchmarks import (
    load_humaneval,
    load_mbpp,
    load_humaneval_x_py2java,
)

### 2.2 Per-benchmark scorers

Each scorer takes a model's generated text and returns `{passed, score, error}`.

In [ ]:
def score_humaneval(row, generated):
    # the model returns the COMPLETE program in a ```python``` block.
    # Use the extracted block as-is; do NOT concatenate row['prompt'].
    program = prompts.extract_python_body(generated)
    full = program + "\n\n" + row['test'] + f"\n\ncheck({row['entry_point']})\n"
    res = execution.run_python(full)
    return {'score': 1.0 if res['passed'] else 0.0, 'passed': res['passed'], 'error': res['error']}

def score_mbpp(row, generated):
    #  the model returns the FULL function (def + body); append the hidden tests.
    program = prompts.extract_python_body(generated)
    full = program + "\n\n" + "\n".join(row['tests']) + "\n"
    res = execution.run_python(full)
    return {'score': 1.0 if res['passed'] else 0.0, 'passed': res['passed'], 'error': res['error']}

def score_humaneval_x(row, generated):
    # the model returns the COMPLETE Java solution in a ```java``` block.
    # Use the extracted block as Solution.java directly; do NOT prepend row['java_prompt'].
    java_source = prompts.extract_java_body(generated)
    res = execution.run_java(java_source, row['java_test'])
    return {'score': 1.0 if res['passed'] else 0.0, 'passed': res['passed'], 'error': res['error']}

### 2.3 Generate + score across one benchmark for one model

In [ ]:
def run_benchmark(model_size, tokenizer, model, benchmark_name, rows):
    out = []
    for row in tqdm(rows, desc=f'{model_size} | {benchmark_name}'):
        if benchmark_name == 'humaneval':
            prompt = prompts.build_completion_prompt(tokenizer, {'prompt': row['prompt']})
        elif benchmark_name == 'mbpp':
            prompt = prompts.build_completion_prompt(tokenizer, {'prompt': row['prompt']})
        elif benchmark_name == 'humaneval_x_py2java':
            prompt = prompts.build_translation_prompt(tokenizer, row['python_source'],
                                                          row['java_declaration'])
        else:
            raise ValueError(benchmark_name)

        gen = prompts.generate(tokenizer, model, prompt, config.DECODING_GREEDY)

        if benchmark_name == 'humaneval':
            r = score_humaneval(row, gen)
        elif benchmark_name == 'mbpp':
            r = score_mbpp(row, gen)
        else:
            r = score_humaneval_x(row, gen)

        out.append({'model': model_size, 'benchmark': benchmark_name, 'problem_id': row['id'],
                    'generated': gen[:2000], **r})
    return out

In [ ]:
# load all benchmarks once
benches = {
    'humaneval': load_humaneval(),
    'mbpp': load_mbpp(),
    'humaneval_x_py2java': load_humaneval_x_py2java(),
}
for name, rows in benches.items():
    print(f'{name}: {len(rows)} problems')

humaneval: 50 problems
mbpp: 50 problems
humaneval_x_py2java: 50 problems


### 2.4 Run Qwen-1.5B (fp16) — all 3 benchmarks

In [ ]:
def _run_at_size(size, benches):
    tok, model = models.load_qwen(size)
    out = []
    try:
        for bn, rows in benches.items():
            if not rows:
                print(f'Skipping {bn} (empty)')
                continue
            out.extend(run_benchmark(size, tok, model, bn, rows))
    finally:
        del tok, model
        gc.collect()
        torch.cuda.empty_cache()
    return out

results_15 = _run_at_size('1.5b', benches)
free, total = torch.cuda.mem_get_info()
print(f'{len(results_15)} rows from 1.5B | GPU free after unload: {free/1e9:.2f} / {total/1e9:.2f} GB')

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

1.5b | humaneval:   0%|          | 0/50 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


1.5b | mbpp:   0%|          | 0/50 [00:00<?, ?it/s]

1.5b | humaneval_x_py2java:   0%|          | 0/50 [00:00<?, ?it/s]

150 rows from 1.5B | GPU free after unload: 9.32 / 15.64 GB


### 2.5 Run Qwen-7B (4-bit nf4) — all 3 benchmarks at K=0

4-bit nf4 fits the 7B in ~6 GB on T4. Quality hit is small but real (proposal §4 Risk #4).

In [ ]:
tok7, m7 = models.load_qwen('7b')
results_7 = []
for bn, rows in benches.items():
    if not rows:
        continue
    results_7.extend(run_benchmark('7b', tok7, m7, bn, rows))
models.unload(m7)
print(f'{len(results_7)} rows from 7B')

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

7b | humaneval:   0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


7b | mbpp:   0%|          | 0/50 [00:00<?, ?it/s]

7b | mbpp:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
all_rows = results_15 + results_7

### 2.6 Save + summarize

In [ ]:
df = pd.DataFrame(all_rows)
for col in ('passed', 'score', 'error'):
    if col not in df.columns:
        df[col] = None
out_path = config.RESULTS_DIR / 'baselines.csv'
df.to_csv(out_path, index=False)
print(f'Saved {len(df)} rows to {out_path}')

summary = df.groupby(['model', 'benchmark'])['score'].agg(['count', 'mean']).reset_index()
summary.columns = ['model', 'benchmark', 'n', 'mean_score']
print(summary.to_string(index=False))

## Part 1–2 complete (Week 1 baselines)

**Sanity ranges (full run, not smoke mode):**
- Qwen-1.5B HumanEval pass@1 ≈ 0.45–0.55
- Qwen-7B (4-bit) HumanEval pass@1 ≈ 0.80–0.86 (vendor reports 0.88 fp16)
- HumanEval-X Java is harder; lower on both models

**Continue below → Part 3 (mentor dataset extension task).**

## Part 3 — Mentor task: Dataset extension (NL + PL1 + PL2)

**Goal:** Every benchmark row should contain all three components:

| Dataset | Already had | We generate | Validation |
|---------|-------------|-------------|------------|
| HumanEval | NL + PL1 (Python) | **PL2** (Java) | PL1 passes Python tests; PL2 passes HumanEval-X Java tests |
| MBPP | NL + PL1 (Python) | **PL2** (Java) | PL1 passes asserts; PL2 compiles |
| HumanEval-X | PL1 + PL2 | **NL** (English) | NL → regenerate PL1' & PL2'; compare output + AST |

**Feedback loop:** up to 3 attempts per row; set `pl2_valid` / `nl_valid` flag.

**Glossary:** NL = English, PL1 = Python, PL2 = Java.

In [ ]:
# 3.1 — extension settings (must be in project dir on sys.path)
import os
import sys
import importlib
import gc

import torch

PROJECT_DIR = os.environ.get('CODEGEN_PROJECT_DIR', '/content/drive/MyDrive/codegen_week1')
os.environ['CODEGEN_PROJECT_DIR'] = PROJECT_DIR
os.environ['CODEGEN_DATA_DIR'] = PROJECT_DIR
%cd {PROJECT_DIR}
# %cd alone is NOT enough in Colab — must add project root to sys.path
if PROJECT_DIR in sys.path:
    sys.path.remove(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
# Clear stale lib cache from earlier imports (before you uploaded ast_compare.py)
for _k in list(sys.modules):
    if _k == 'lib' or _k.startswith('lib.'):
        del sys.modules[_k]

# verify uploaded mentor-task files (upload to Drive: codegen_week1/lib/)
_required = [
    'config.py',
    'lib/__init__.py',
    'lib/ast_compare.py',
    'lib/extend.py',
    'lib/benchmarks.py',
]
_missing = [p for p in _required if not os.path.isfile(os.path.join(PROJECT_DIR, p))]
if _missing:
    raise FileNotFoundError(
        'Missing on Drive under codegen_week1:\n  ' + '\n  '.join(_missing) +
        '\n\nUpload lib/ast_compare.py, lib/extend.py, and updated config.py to My Drive/codegen_week1/'
    )
print('Project dir OK:', PROJECT_DIR)
print('lib files:', os.listdir(os.path.join(PROJECT_DIR, 'lib')))

# First dry run: 5 rows per dataset. Full run: set CODEGEN_EXTEND_SAMPLE=50
os.environ.setdefault('CODEGEN_EXTEND_SAMPLE', '5')
os.environ.setdefault('CODEGEN_EXTEND_RETRIES', '3')

import config
importlib.reload(config)
config.ensure_dirs()

# Use submodule imports — NOT "from lib import ast_compare" (__init__.py does not export them)
import lib.ast_compare
import lib.extend
import lib.benchmarks

from lib.benchmarks import (
    load_humaneval_for_extension,
    load_mbpp_for_extension,
    load_humaneval_x_for_extension,
)
from lib.extend import run_extension, save_extended, summarize_extension
from lib import models

he_ext = load_humaneval_for_extension()
mbpp_ext = load_mbpp_for_extension()
hex_ext = load_humaneval_x_for_extension()
print('Extension rows:', len(he_ext), 'humaneval |', len(mbpp_ext), 'mbpp |', len(hex_ext), 'humaneval-x')
print('Output dir:', config.EXTENDED_DIR)

In [ ]:
# 3.2 — load generation model (1.5B is faster for extension; switch to '7b' if desired)
gc.collect()
torch.cuda.empty_cache()

EXTEND_MODEL_SIZE = os.environ.get('CODEGEN_EXTEND_MODEL', '1.5b')
tok_ext, model_ext = models.load_qwen(EXTEND_MODEL_SIZE)
print('Extension model:', EXTEND_MODEL_SIZE)

In [ ]:
# 3.3 — preview one row per dataset
print('--- HumanEval (has NL+PL1, need PL2) ---')
print('id:', he_ext[0]['id'])
print('NL:', he_ext[0]['nl'][:200], '...')
print('has java_test:', bool(he_ext[0].get('java_test')))

print('\n--- MBPP (has NL+PL1, need PL2) ---')
print('id:', mbpp_ext[0]['id'])
print('NL:', mbpp_ext[0]['nl'][:200], '...')

print('\n--- HumanEval-X (has PL1+PL2, need NL) ---')
print('id:', hex_ext[0]['id'])
print('PL1 head:', hex_ext[0]['pl1'][:120], '...')
print('PL2 head:', hex_ext[0]['pl2'][:120], '...')

In [ ]:
# 3.4 — run mentor extension (this may take a while)
extended_rows = run_extension(
    tok_ext,
    model_ext,
    humaneval_rows=he_ext,
    mbpp_rows=mbpp_ext,
    hex_rows=hex_ext,
)
print(f'Extended {len(extended_rows)} rows')
print(summarize_extension(extended_rows))

In [ ]:
# 3.5 — save unified dataset + show sample flags
import pandas as pd

out_jsonl = save_extended(extended_rows)
print('Saved:', out_jsonl)

df_ext = pd.DataFrame(extended_rows)
cols = [c for c in ['dataset', 'id', 'pl2_valid', 'nl_valid', 'pl2_attempts', 'nl_attempts'] if c in df_ext.columns]
print(df_ext[cols].head(10).to_string(index=False))

# unload extension model
del tok_ext, model_ext
gc.collect()
torch.cuda.empty_cache()

### Part 3 notes

- **HumanEval / MBPP:** generated `pl2` from `nl` + `pl1`; `pl2_valid=True` when Java tests pass (HumanEval) or Java compiles (MBPP).
- **HumanEval-X:** generated `nl` from `pl1` + `pl2`; `nl_valid=True` when NL can regenerate matching PL1 (output+AST) and passing PL2 (Java tests).
- **Scale up:** set `os.environ['CODEGEN_EXTEND_SAMPLE'] = '50'` (or remove for full 164).
- **Output file:** `extended/unified_dataset.jsonl` on Google Drive under your `CODEGEN_DATA_DIR`.